In [6]:
import pandas as pd
import numpy as np

# ── Load Data ──────────────────────────────────────────────
df = pd.read_csv('../data/dataset.csv')

# ── Clean column names & values ────────────────────────────
df.columns = df.columns.str.strip()
for col in df.columns:
    if col != 'Disease':
        df[col] = df[col].str.strip() if df[col].dtype == 'object' else df[col]

# ── Get all unique symptoms ────────────────────────────────
symptom_cols = [c for c in df.columns if c.startswith('Symptom')]
all_symptoms = pd.unique(df[symptom_cols].values.ravel())
all_symptoms = [s.strip() for s in all_symptoms if isinstance(s, str)]
all_symptoms = sorted(set(all_symptoms))
print(f"Total unique symptoms: {len(all_symptoms)}")

# ── Convert to one-hot encoded format ─────────────────────
def encode_row(row):
    present = set()
    for col in symptom_cols:
        val = row[col]
        if isinstance(val, str):
            present.add(val.strip())
    return pd.Series(
        {s: 1 if s in present else 0 for s in all_symptoms}
    )

print("Encoding symptoms... (takes 30 seconds)")
encoded = df.apply(encode_row, axis=1)
encoded['prognosis'] = df['Disease'].str.strip()

print(f"Final shape: {encoded.shape}")
print(encoded['prognosis'].value_counts())
print("✅ Done!")

Total unique symptoms: 131
Encoding symptoms... (takes 30 seconds)
Final shape: (4920, 132)
prognosis
Fungal infection                           120
Allergy                                    120
GERD                                       120
Chronic cholestasis                        120
Drug Reaction                              120
Peptic ulcer diseae                        120
AIDS                                       120
Diabetes                                   120
Gastroenteritis                            120
Bronchial Asthma                           120
Hypertension                               120
Migraine                                   120
Cervical spondylosis                       120
Paralysis (brain hemorrhage)               120
Jaundice                                   120
Malaria                                    120
Chicken pox                                120
Dengue                                     120
Typhoid                                    120
hepat

In [7]:
encoded.to_csv('../data/symptoms_clean.csv', index=False)
print("✅ Clean dataset saved to data/symptoms_clean.csv")

✅ Clean dataset saved to data/symptoms_clean.csv


In [8]:
from sklearn.preprocessing import LabelEncoder

# Load clean dataset
df_clean = pd.read_csv('../data/symptoms_clean.csv')

# Separate features and target
X = df_clean.drop('prognosis', axis=1)
y = df_clean['prognosis']

# Encode target
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Total features (symptoms):", X.shape[1])
print("Total diseases:", len(le.classes_))
print("Diseases:", list(le.classes_))

Total features (symptoms): 131
Total diseases: 41
Diseases: ['(vertigo) Paroymsal  Positional Vertigo', 'AIDS', 'Acne', 'Alcoholic hepatitis', 'Allergy', 'Arthritis', 'Bronchial Asthma', 'Cervical spondylosis', 'Chicken pox', 'Chronic cholestasis', 'Common Cold', 'Dengue', 'Diabetes', 'Dimorphic hemmorhoids(piles)', 'Drug Reaction', 'Fungal infection', 'GERD', 'Gastroenteritis', 'Heart attack', 'Hepatitis B', 'Hepatitis C', 'Hepatitis D', 'Hepatitis E', 'Hypertension', 'Hyperthyroidism', 'Hypoglycemia', 'Hypothyroidism', 'Impetigo', 'Jaundice', 'Malaria', 'Migraine', 'Osteoarthristis', 'Paralysis (brain hemorrhage)', 'Peptic ulcer diseae', 'Pneumonia', 'Psoriasis', 'Tuberculosis', 'Typhoid', 'Urinary tract infection', 'Varicose veins', 'hepatitis A']


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples :", X_test.shape[0])

Training samples: 3936
Testing samples : 984


In [10]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
model.fit(X_train, y_train)
print("✅ Model trained successfully!")

✅ Model trained successfully!


In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Accuracy: {accuracy * 100:.2f}%")
print(classification_report(
    y_test, y_pred,
    target_names=le.classes_
))

✅ Accuracy: 100.00%
                                         precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00        18
                                   AIDS       1.00      1.00      1.00        30
                                   Acne       1.00      1.00      1.00        24
                    Alcoholic hepatitis       1.00      1.00      1.00        25
                                Allergy       1.00      1.00      1.00        24
                              Arthritis       1.00      1.00      1.00        23
                       Bronchial Asthma       1.00      1.00      1.00        33
                   Cervical spondylosis       1.00      1.00      1.00        23
                            Chicken pox       1.00      1.00      1.00        21
                    Chronic cholestasis       1.00      1.00      1.00        15
                            Common Cold       1.00      1.00      1.00        23
       

In [12]:
import joblib

joblib.dump(model, '../model/disease_model.pkl')
joblib.dump(le,    '../model/label_encoder.pkl')
joblib.dump(list(X.columns), '../model/symptom_columns.pkl')

print("✅ Model saved!")
print("✅ Encoder saved!")
print("✅ Symptom columns saved!")

✅ Model saved!
✅ Encoder saved!
✅ Symptom columns saved!
